# Studio di MERGE
basato su cleaning 4
## Inizializzazione ed Import

In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
from MergerTools import *
import pandas as pd
import os

client = DatalakeClient()
mergeTools = MergerTools()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake


**Mixed info**\
'ADNIMERGE', \
'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', --> have CSF

**Single Cofactor**\
'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES'

**Volumes**\
'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSL', 'UCSDVOL', 'UPENN_ROI_MARS',\
'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSX51_ADNI1_3T'  --> just partial immages segmentation

**CSF**\
'UPENNBIOMK_ADNIDIAN_ES_2017', 'UPENNBIOMK_ROCHE_ELECSYS', 'EUROIMMUN', 'FUJIREBIOABETA', 'SALADAX_BIOMEDICAL', 'MESOSCALE', 'UPENNBIOMK_MASTER', 'UPENN_2DUPLC_CRM', 

In [2]:
file_codes =['ADNIMERGE', 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 
                'UCSFFSL', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSX51_ADNI1_3T']
search = client.query_files(
    query={'custom.level' : 'cleaned_04', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

print(len(zip_files))

10


In [3]:
cathegory = 'volumes'

In [ ]:
dfs = {}
df_names = {}
df_code = []
for idx, (file_name, df_raw) in enumerate(zip_files.items()):
    df_copy = df_raw.copy(deep=True)
    len_pre = len(df_copy.columns)
    # get the sub_df focusing on the cathegory chosen
    df_copy = mergeTools.filter_df_cathegory(df_copy, cathegory)
    len_post = len(df_copy.columns)
    # get the common columns among all the dfs
    if idx == 0:
        dfs_columns = set(df_copy.columns)
    else: 
        dfs_columns &= set(df_copy.columns)
    # Ensure EXAMDATE in date format and correct order of the dfs
    df_copy['EXAMDATE'] = pd.to_datetime(df_copy['EXAMDATE'])
    df_copy = df_copy.sort_values(by=['RID', 'EXAMDATE']).reset_index(drop=True)
    if 'FSVERSION' in df_copy.columns:
        df_copy['FSVERSION'] = df_copy['FSVERSION'].astype(str)
    # aggiornamento liste e dizionari
    dfs[f"df_{idx}"] = df_copy  
    df_names[f"df_{idx}"] = file_name 
    df_code.append(f"df_{idx}")
    # definizione variabile df
    globals()[f"df_{idx}"] = df_copy
    print(idx, '--->', file_name, '\t\t\t### ', len_post, '/', len_pre)

time_buffer = pd.Timedelta(days=80)

0 ---> ADNIMERGE_25Jul2025_04.csv 			###  14 / 41
1 ---> UCSFFSX_11_02_15_11Aug2025_04.csv 			###  14 / 14
2 ---> UCSFFSX7_11Aug2025_04.csv 			###  15 / 15
3 ---> UCSFFSX6_11Aug2025_04.csv 			###  14 / 14
4 ---> UCSFFSX51_11_08_19_11Aug2025_04.csv 			###  14 / 14
5 ---> UCSFFSX51_ADNI1_3T_02_01_16_11Aug2025_04.csv 			###  14 / 14
6 ---> UCSFFSL51ALL_08_01_16_11Aug2025_04.csv 			###  14 / 14
7 ---> UCSFFSL51_03_01_22_11Aug2025_04.csv 			###  14 / 14
8 ---> UCSFFSL51Y1_08_01_16_11Aug2025_04.csv 			###  14 / 14
9 ---> UCSFFSL_02_01_16_11Aug2025_04.csv 			###  14 / 14


In [5]:
x = 0
for df_x in dfs.values():
    print(x, df_x['FSVERSION'].unique())
    x += 1

0 [4.3 nan 5.1 6. ]
1 [4.3]
2 ['7.4.1']
3 [6.]
4 [5.1]
5 [5.1]
6 [5.1]
7 [5.1]
8 [5.1]
9 [4.4]


# Confronto stessi RID  ==> RID - EXAMDATE identici tra file

In [6]:

subj_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID'])
print("righe con stessi ####### RID:")
display(subj_matrix)
        
subj_date_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID', 'EXAMDATE'], time_buffer=time_buffer)
print("righe con stessi ####### RID-EXAMDATE: --> time_buffer=", time_buffer)
display(subj_date_matrix)

if set(['FSVERSION', 'IMAGEUID']).issubset(set(dfs_columns)):
    subj_viscode_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID', 'EXAMDATE', 'FSVERSION','IMAGEUID'])
    print("righe con stessi ####### RID-EXAMDATE-FSVERSION-IMAGEUID: --> time_buffer=", time_buffer)
    display(subj_viscode_matrix)

righe con stessi ####### RID:


,df_0,df_1,df_2,df_3,df_4,df_5,df_6,df_7,df_8,df_9
df_0,2409,,,,,,,,,
df_1,818,842,,,,,,,,
df_2,281,11,807,,,,,,,
df_3,1008,64,281,1122,,,,,,
df_4,980,78,71,313,1049,,,,,
df_5,138,138,3,12,13,138,,,,
df_6,113,1,1,12,113,1,113,,,
df_7,662,25,48,209,662,5,89,662,,
df_8,347,7,25,122,347,2,30,292,347,
df_9,739,739,11,62,77,132,1,25,7,739


righe con stessi ####### RID-EXAMDATE: --> time_buffer= 80 days 00:00:00


,df_0,df_1,df_2,df_3,df_4,df_5,df_6,df_7,df_8,df_9
df_0,11458,,,,,,,,,
df_1,4022,4076,,,,,,,,
df_2,0,0,847,,,,,,,
df_3,1958,0,0,2222,,,,,,
df_4,3645,5,0,0,4038,,,,,
df_5,431,411,0,0,0,435,,,,
df_6,381,0,0,0,401,0,424,,,
df_7,2869,0,0,0,2965,0,357,3123,,
df_8,1161,1,0,0,1259,0,100,1071,1283,
df_9,3330,3333,0,0,3,396,0,0,0,3354


righe con stessi ####### RID-EXAMDATE-FSVERSION-IMAGEUID: --> time_buffer= 80 days 00:00:00


,df_0,df_1,df_2,df_3,df_4,df_5,df_6,df_7,df_8,df_9
df_0,11458,,,,,,,,,
df_1,1508,4076,,,,,,,,
df_2,0,0,847,,,,,,,
df_3,536,0,0,2222,,,,,,
df_4,1501,0,0,0,4038,,,,,
df_5,0,0,0,0,0,435,,,,
df_6,161,0,0,0,388,0,424,,,
df_7,1235,0,0,0,2812,0,357,3123,,
df_8,461,0,0,0,1166,0,100,1071,1283,
df_9,0,0,0,0,0,0,0,0,0,3354


# Inizio Merge
## Definizione di df_base e Gerarchia di DF da mergiare
Scegliere il file con numero maggiore di soggetti-visite e che ha più elementi con altri df.\
Quindi scegliere con che ordine unire gli altri df, suggerimento da quelli con nessuna/pochissime righe RID-EXAMDATE in comune con gli altri df, e quindi quelli con molte righe in comune a partire da quello con più righe in comune sia con df_base che con gli altri e quindi a seguire. Ma di persè il metodo è arbitrario quindi si può fare come si vuole.

In [8]:
df_base = dfs['df_4'].copy(deep=True)                           #sembra un errore ma questi df sono definiti
idx_add = [ 'df_6', 'df_7', 'df_8', 'df_9','df_1', 'df_2', 'df_3', 'df_5', 'df_0']
merge_contains = ['df_4']
sub_with_match = set()
time_buffer = pd.Timedelta(days=80)
i = 0

## Approfondimento RID-EXAMDATE
1. vedo quante righe ci sono con match ESATTO e quante con TIME BUFFER.

In [46]:
df_add = dfs[idx_add[i]].copy(deep=True)
print(idx_add[i])

df_2


In [47]:
exact_matches, buffer_matches = mergeTools.find_visit_matches(df_base, df_add, buffer_days=time_buffer)
exact_index1, exact_index2 = mergeTools.list_index_visit_matches(exact_matches)
buff_index1, buff_index2 = mergeTools.list_index_visit_matches(buffer_matches)


print('Exact matches: \t', len(exact_index1))
print('Buffered matches: \t', len(buff_index1))

if len(buff_index1) != len(buff_index2):
    print('\n ------>>> ATTENZIONE: indici match con buffer SPAIATI')

if len(buff_index1) != len(buff_index2):
    print('\n ------>>> ATTENZIONE: indici match SPAIATI')

Exact matches: 	 0
Buffered matches: 	 0


In [48]:
columns_in_common = list(df_base.columns.intersection(df_add.columns))
columns_only_base = list(df_base.columns.difference(df_add.columns))
columns_only_add = list(df_add.columns.difference(df_base.columns))

all_index1 = exact_index1.union(buff_index1)
all_index2 = exact_index2.union(buff_index2)

# studio le colonne in comune e non ai due df
print('Le colonne in comune sono:\n', columns_in_common)
print('\n\nLe colonne solo in df_base sono: \n', columns_only_base)
print('\n\nLe colonne solo in df_add sono: \n', columns_only_add)
print('__________________________________________________________________\n\n')

diff_date = False
# verifico ci siano righe matchate con BUFFER
if len(buff_index1) == len(buff_index2) and len(buff_index1) > 0:    
    diff_date = True
    # verifico se le righe matchate sono TUTTE matchate con BUFFER
    if all(all_index1) == len(all_index2) and all_index1 == buff_index1 and all_index2 == buff_index2:
        print(f'all maches have a buffer, tot: {len(all_index1)} matches\n\n====================================> Renamen\'EXAMDATE\' column\n')
    else:
        print(f'There are {len(buff_index1)} match with buffer\nThere are {len(all_index1)-len(buff_index1)} match exact\nOver {len(all_index1)} total matches\n\n====================================> SHOULD \'EXAMDATE\' column be renamed?\n')
    

# ci sono solo match ESATTI
elif len(buff_index1) == len(buff_index2) and len(buff_index1) == 0 and len(all_index1) > 0:
    print('JUST exact matches')
elif len(buff_index1) == len(buff_index2) and len(buff_index1) == 0 and len(all_index1) == 0:
    print('NO matches')


Le colonne in comune sono:
 ['COHORT', 'RID', 'VISCODE', 'VISIT_MONTH', 'EXAMDATE', 'IMAGEUID', 'STATUS', 'FSVERSION', 'ICV%ICV', 'Fusiform%ICV', 'Hippocampus%ICV', 'Ventricles%ICV', 'Entorhinal%ICV', 'MidTemp%ICV', 'FLDSTRENG']


Le colonne solo in df_base sono: 
 []


Le colonne solo in df_add sono: 
 []
__________________________________________________________________


NO matches


In [49]:
if diff_date:
    print('df_add --> ', df_names[idx_add[i]], '\ndf_base --> ', [df_names[x] for x in merge_contains])
    col_list = list(df_base.columns) + [c for c in df_add.columns if c not in df_base.columns]
    
    temp_merge = mergeTools.create_temp_merge(df_base, df_add, buff_index1, buff_index2, col_list=col_list)
    diff = temp_merge['EXAMDATE_1']-temp_merge['EXAMDATE_2']
    display(diff[diff != pd.Timedelta(days=0)])
    print(len(diff[diff != pd.Timedelta(days=0)]))
    display(temp_merge.loc[diff[diff != pd.Timedelta(days=0)].index])
    

In [ ]:
modify_examdate = False #True   #False
if modify_examdate:
    df_add.loc[buff_index2, 'EXAMDATE'] = df_base.loc[buff_index1, 'EXAMDATE'].values
    display(df_add.loc[buff_index2]['EXAMDATE'])

294    2010-10-04
294    2010-10-04
3155   2010-03-18
3155   2010-03-18
3345   2010-03-04
3599   2010-02-16
3599   2010-02-16
Name: EXAMDATE, dtype: datetime64[ns]

## Studio Colonne in comune per righe che matchano
1) Se ci sono righe che machano tra i due df allora identifico altre colonne in comune ai due df.

2) Faccio merge tra i due df escludendo i soggetti che hanno visite metchate tra i 2 df (righe). --> merge_base solo aggiunta di soggetti nuovi.

3) Quindi se ci sono colonne in comune e righe che matchano faccio merge soggetto per soggetto (tra i soggetti con  visite in entrambi i df).\
Aggiungo qusti merge di singoli soggetti al resto del merge_base.


Così ottengo Merge finale.

In [57]:
type(df_1.loc[0,'FSVERSION'])

numpy.float64

In [50]:
df_merged = mergeTools.get_merged_df(df_base, df_add, cathegory=cathegory)

### merging volumes


ValueError: You are trying to merge on float64 and object columns for key 'FSVERSION'. If you wish to proceed you should use pd.concat

In [45]:
df_base = df_merged.copy(deep=True)
merge_contains.append(idx_add[i])
#prepare for next merge
i += 1
print(f'il merg contine i seguenti df: {merge_contains}')
if i <= len(idx_add)-1:
    print(f'il prossimo df da unire è: {idx_add[i]}\n\n ===> torna al capitolo: "Approfondimento RID-EXAMDATE"')
else:
    print('FINISHED MERGE!!!!!')

il merg contine i seguenti df: ['df_4', 'df_6', 'df_7', 'df_8', 'df_9', 'df_1']
il prossimo df da unire è: df_2

 ===> torna al capitolo: "Approfondimento RID-EXAMDATE"


# Verifiche specifiche

In [ ]:
col_list = list(df_base.columns) + [c for c in df_add.columns if c not in df_base.columns]
temp_merge = mergeTools.create_temp_merge(df_base, df_add, all_index1, all_index2, rid=1225 , col_list=col_list)
merged_sub_df = mergeTools.merge_paired_rows_rid_specific(df_base, df_add, all_index1, all_index2, ref_col, subject_id=1225 )

    

In [ ]:
temp_merge

In [ ]:
merged_sub_df

# altro